In [32]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Setup
os.makedirs("models", exist_ok=True)
df = pd.read_csv('Gold Price.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Feature Engineering
df['Lag1'] = df['Price'].shift(1)
df['Lag2'] = df['Price'].shift(2)
df['MA7'] = df['Price'].rolling(7).mean()
df['Price_Diff'] = df['Price'] - df['Lag1']
df = df.dropna()

# Split
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]
X_train, y_train = train[['Lag1', 'Lag2', 'MA7']], train['Price_Diff']
X_test, y_test = test[['Lag1', 'Lag2', 'MA7']], test['Price']

# Model
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
pred = X_test['Lag1'].values + rf.predict(X_test)

# Evaluation
print("=== Random Forest Evaluation ===")
print(f"MAE  : {mean_absolute_error(y_test, pred):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred)):.4f}")
print(f"R2   : {r2_score(y_test, pred):.4f}")
print(f"MAPE : {np.mean(np.abs((y_test - pred) / y_test)) * 100:.4f}")

joblib.dump(rf, "models/random_forest.pkl")

=== Random Forest Evaluation ===
MAE  : 775.1643
RMSE : 1041.7130
R2   : 0.9974
MAPE : 0.9086


['models/random_forest.pkl']

In [33]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import GradientBoostingRegressor

# Model
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
pred = X_test['Lag1'].values + gb.predict(X_test)

# Evaluation
print("=== Gradient Boosting Evaluation ===")
print(f"MAE  : {mean_absolute_error(y_test, pred):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred)):.4f}")
print(f"R2   : {r2_score(y_test, pred):.4f}")
print(f"MAPE : {np.mean(np.abs((y_test - pred) / y_test)) * 100:.4f}")

joblib.dump(gb, "models/gradient_boosting.pkl")

=== Gradient Boosting Evaluation ===
MAE  : 922.9305
RMSE : 1172.5091
R2   : 0.9968
MAPE : 1.0936


['models/gradient_boosting.pkl']

In [34]:
import pandas as pd
import numpy as np
import joblib
import os
import random

# --- FIXED SEED BLOCK START ---
# Set a seed value
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)
# ------------------------------

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Ensure the models directory exists
os.makedirs("models", exist_ok=True)

# 1. Scaling & Windowing
scaler = MinMaxScaler()
scaled_prices = scaler.fit_transform(df[['Price']])
joblib.dump(scaler, 'models/scaler.pkl')

def create_window(data, window=60):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_lstm, y_lstm = create_window(scaled_prices)
X_lstm = X_lstm.reshape(X_lstm.shape[0], X_lstm.shape[1], 1)
l_split = int(len(X_lstm) * 0.8)

# 2. Model Architecture
lstm = Sequential([
    Input(shape=(60, 1)),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(1)
])

# 3. Compile and Train
lstm.compile(optimizer='adam', loss='mse')
# Set verbose=1 if you want to see the progress bar during training
lstm.fit(X_lstm[:l_split], y_lstm[:l_split], epochs=30, batch_size=32, verbose=1)

# 4. Prediction
preds = scaler.inverse_transform(lstm.predict(X_lstm[l_split:])).flatten()
actual = scaler.inverse_transform(y_lstm[l_split:].reshape(-1, 1)).flatten()

# 5. Evaluation with MAPE
print("=== LSTM Evaluation ===")
print(f"MAE   : {mean_absolute_error(actual, preds):.4f}")
print(f"RMSE  : {np.sqrt(mean_squared_error(actual, preds)):.4f}")
print(f"R2    : {r2_score(actual, preds):.4f}")
print(f"MAPE  : {np.mean(np.abs((actual - preds) / actual)) * 100:.4f}")

# 6. Save Model
lstm.save("models/lstm_model.keras")

Epoch 1/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 0.0014
Epoch 2/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 4.8261e-05
Epoch 3/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 4.6319e-05
Epoch 4/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 4.5242e-05
Epoch 5/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - loss: 4.4017e-05
Epoch 6/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 4.2503e-05
Epoch 7/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 4.1689e-05
Epoch 8/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 4.0970e-05
Epoch 9/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - loss: 4.0192e-05
Epoch 10/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 3.9444e-05
Epoch 11/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - loss: 3.8373e-05
Epoch 12/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 3.7026e-05
Epoch 13/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - loss: 3.5660e-05
Epoch 14/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - loss: 3.4495e-05
Epoch 15/30
76/76 ━

In [35]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import load_model

# Feature from LSTM
lstm_full = scaler.inverse_transform(lstm.predict(X_lstm)).flatten()
hybrid_df = df.iloc[60:].copy()
hybrid_df['LSTM_Pred'] = lstm_full[:len(hybrid_df)]

# Features & Split
X_h = hybrid_df[['Lag1', 'Lag2', 'MA7', 'LSTM_Pred']]
y_h = hybrid_df['Price'] - hybrid_df['Lag1']
h_split = int(len(X_h) * 0.8)

# Model
h_rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
h_rf.fit(X_h.iloc[:h_split], y_h.iloc[:h_split])

# Prediction
pred = X_h.iloc[h_split:]['Lag1'].values + h_rf.predict(X_h.iloc[h_split:])
actual = hybrid_df['Price'].values[h_split:]

# Evaluation
print("=== Hybrid Model Evaluation ===")
print(f"MAE  : {mean_absolute_error(actual, pred):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(actual, pred)):.4f}")
print(f"R2   : {r2_score(actual, pred):.4f}")
print(f"MAPE : {np.mean(np.abs((actual - pred) / actual)) * 100:.4f}")

joblib.dump(h_rf, "models/hybrid_rf.pkl")

95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step
=== Hybrid Model Evaluation ===
MAE  : 747.2266
RMSE : 1017.6311
R2   : 0.9975
MAPE : 0.8715


['models/hybrid_rf.pkl']